In [1]:
import pandas as pd
import numpy as np
import os
import glob

# 1. Basispfade definieren (Pfade anpassen, falls nötig)
BASE_DIR = r"C:\Users\flori\Desktop\output_scenarios_test\Test Bay"
F1_DIR = os.path.join(BASE_DIR, "Test_Bay_F1", "Extracted_Measurements")
F2_DIR = os.path.join(BASE_DIR, "Test_Bay_F2", "Extracted_Measurements")

# Schwellenwert für Grenzwertverletzung (z.B. 253V bei 230V Nennspannung -> +10%)
# Je nach Netzrichtlinie kann das auch 1.05 p.u. (241.5V) sein.
U_MAX_THRESHOLD = 253.0 

def extract_features(df, prefix):
    """
    Berechnet physikalische Features aus einem Zeitreihen-DataFrame.
    prefix: 'smartmeter_' oder 'ortsnetztrafo_' (um die korrekten Spalten zu finden)
    """
    features = {}
    
    # Finde die entsprechenden Spaltennamen im DataFrame
    # Spannung L1 (als Referenz für Spannungshub etc., man könnte auch L1, L2, L3 aggregieren)
    u_cols = [c for c in df.columns if 'Spannung' in c and ('U1-N' in c or 'L1N' in c)]
    p_col = [c for c in df.columns if 'Wirkleistung' in c and 'gesamt' in c]
    pf_col = [c for c in df.columns if ('Powerfactor' in c or 'Leistungsfaktor' in c) and 'gesamt' in c]
    
    if not u_cols or not p_col:
        return features # Falls Spalten nicht gefunden werden
    
    u_col = u_cols[0]
    p_col = p_col[0]
    
    # Daten in numerische Werte umwandeln (falls nicht schon geschehen)
    df[u_col] = pd.to_numeric(df[u_col], errors='coerce')
    df[p_col] = pd.to_numeric(df[p_col], errors='coerce')
    
    # 1. Spannungshub (U_max - U_min)
    u_max = df[u_col].max()
    u_min = df[u_col].min()
    features[f'{prefix}Spannungshub'] = u_max - u_min
    
    # 2. Varianz der Spannung (Wie unruhig ist das Netz?)
    features[f'{prefix}Spannung_Varianz'] = df[u_col].var()
    
    # 3. Wirk- und Blindleistungs-Verhalten
    p_max = df[p_col].max()
    p_min = df[p_col].min()
    features[f'{prefix}P_max'] = p_max
    features[f'{prefix}P_min'] = p_min
    
    # Leistungsfaktor (cos phi) zum Zeitpunkt der maximalen Wirkleistung
    if pf_col:
        df[pf_col[0]] = pd.to_numeric(df[pf_col[0]], errors='coerce')
        # Finde den Index (Zeitpunkt), an dem P maximal ist
        idx_p_max = df[p_col].idxmax()
        if pd.notna(idx_p_max):
            cos_phi_at_pmax = df.loc[idx_p_max, pf_col[0]]
            features[f'{prefix}CosPhi_bei_P_max'] = cos_phi_at_pmax
    
    # 4. Grenzwertverletzungen (Wie oft/lange war die Spannung zu hoch?)
    # Hier: Anzahl der Messpunkte über dem Schwellenwert
    violation_count = (df[u_col] > U_MAX_THRESHOLD).sum()
    features[f'{prefix}Spannung_Grenzwertverletzungen'] = violation_count

    return features

def process_scenarios():
    all_features = []
    
    # Wir iterieren durch die 19 Szenarien
    for scenario_idx in range(1, 20):
        scenario_data = {'Szenario': scenario_idx}
        
        # Suchmuster für die Dateien (unterstützt CSV oder Excel)
        # Angenommen, die Dateien heißen z.B. "Szenario_1.csv" oder "output_1_...csv"
        f1_pattern = os.path.join(F1_DIR, f"*{scenario_idx}_*.csv") 
        f2_pattern = os.path.join(F2_DIR, f"*{scenario_idx}_*.csv")
        
        f1_files = glob.glob(f1_pattern)
        f2_files = glob.glob(f2_pattern)
        
        # --- Smart Meter Daten verarbeiten (F1) ---
        if f1_files:
            try:
                # Dezimaltrennzeichen Komma beachten
                df_f1 = pd.read_csv(f1_files[0], sep=',', decimal=',') 
                f1_features = extract_features(df_f1, prefix='SM_')
                scenario_data.update(f1_features)
            except Exception as e:
                print(f"Fehler beim Lesen von F1 Szenario {scenario_idx}: {e}")
        
        # --- Trafodaten verarbeiten (F2) ---
        if f2_files:
            try:
                df_f2 = pd.read_csv(f2_files[0], sep=',', decimal=',')
                f2_features = extract_features(df_f2, prefix='Trafo_')
                scenario_data.update(f2_features)
            except Exception as e:
                print(f"Fehler beim Lesen von F2 Szenario {scenario_idx}: {e}")
                
        # Nur hinzufügen, wenn auch Daten gefunden wurden
        if len(scenario_data) > 1:
            all_features.append(scenario_data)
            
    # Aus der Liste von Dictionaries einen sauberen DataFrame erstellen
    features_df = pd.DataFrame(all_features)
    return features_df

if __name__ == "__main__":
    print("Starte Feature Extraction...")
    final_dataset = process_scenarios()
    
    if not final_dataset.empty:
        print(final_dataset.head())
        
        # Speichere das Ergebnis für das spätere Machine Learning Modell (PCA etc.)
        output_path = os.path.join(BASE_DIR, "Extracted_Physical_Features.csv")
        final_dataset.to_csv(output_path, index=False, sep=';', decimal=',')
        print(f"\nErfolgreich gespeichert unter: {output_path}")
        print(f"Die neue Tabelle hat {final_dataset.shape[0]} Zeilen (Samples) und {final_dataset.shape[1]} Spalten (Features).")
    else:
        print("Es konnten keine Daten verarbeitet werden. Bitte überprüfe die Dateinamen und Pfade.")

Starte Feature Extraction...
   Szenario  SM_Spannungshub  SM_Spannung_Varianz    SM_P_max     SM_P_min  \
0         1         5.000015             1.584775  -58.299999 -5317.899902   
1         2         4.199997             1.153925 -532.900024 -5877.899902   
2         3         7.000000             2.499918 -365.299988 -6224.200195   
3         4         5.899994             3.319465 -201.500000 -6299.000000   
4         5         8.099991             4.337319  108.699997 -6808.600098   

   SM_CosPhi_bei_P_max  SM_Spannung_Grenzwertverletzungen  Trafo_Spannungshub  \
0               -0.867                                 44            1.849075   
1               -0.993                                 39            1.938553   
2               -0.994                                 40            7.106094   
3               -0.966                                 38            2.186813   
4                0.945                                 39            3.673798   

   Trafo_Spannu

In [3]:
import pandas as pd
import numpy as np
import os
import glob
import csv

# Basispfad definieren - ggf. anpassen
BASE_DIR = r"C:\Users\flori\Desktop\output_scenarios_test\Test Bay"
U_MAX_THRESHOLD = 253.0 # Grenzwert für Spannung (230V + 10%)

def process_file(file_path, prefix):
    # 1. Datei exakt im Originalformat einlesen
    # (Trennzeichen: Komma, Dezimalzeichen: Komma)
    df = pd.read_csv(file_path, sep=',', decimal=',')
    
    # 2. Relevante Spalten für die Berechnung identifizieren
    u_cols = [c for c in df.columns if 'Spannung' in c and ('U1-N' in c or 'L1N' in c)]
    p_cols = [c for c in df.columns if 'Wirkleistung' in c and 'gesamt' in c]
    pf_cols = [c for c in df.columns if ('Powerfactor' in c or 'Leistungsfaktor' in c) and 'gesamt' in c]

    # Falls die Datei nicht die erwarteten Spalten hat, unverändert zurückgeben
    if not u_cols or not p_cols:
        return df
        
    u_col = u_cols[0]
    p_col = p_cols[0]

    # Temporär für die Berechnung sicherstellen, dass die Werte numerisch sind
    u_series = pd.to_numeric(df[u_col], errors='coerce')
    p_series = pd.to_numeric(df[p_col], errors='coerce')
    
    # --- 3. Physikalische Features berechnen ---
    u_max = u_series.max()
    u_min = u_series.min()
    
    # Namen der Features mit dem gewünschten Prefix (smartmeter_ oder ortsnetztrafo_)
    features = {
        f'{prefix}Spannungshub': u_max - u_min,
        f'{prefix}Spannung_Varianz': u_series.var(),
        f'{prefix}P_max': p_series.max(),
        f'{prefix}P_min': p_series.min(),
        f'{prefix}Grenzwertverletzungen': (u_series > U_MAX_THRESHOLD).sum()
    }
    
    # Cos Phi zum Zeitpunkt der maximalen Wirkleistung
    if pf_cols:
        pf_series = pd.to_numeric(df[pf_cols[0]], errors='coerce')
        idx_p_max = p_series.idxmax()
        if pd.notna(idx_p_max):
            features[f'{prefix}CosPhi_bei_P_max'] = pf_series.loc[idx_p_max]
        else:
            features[f'{prefix}CosPhi_bei_P_max'] = np.nan

    # --- 4. Berechnete Werte als neue Spalten anfügen ---
    for feat_name, feat_value in features.items():
        df[feat_name] = feat_value
        
    return df

def run_extraction():
    # Hier werden die exakten gewünschten Präfixe zugewiesen
    folders = {
        "Test_Bay_F1": "smartmeter_",
        "Test_Bay_F2": "ortsnetztrafo_"
    }

    for folder_name, prefix in folders.items():
        search_path = os.path.join(BASE_DIR, folder_name, "Extracted_Measurements", "*.csv")
        files = glob.glob(search_path)
        
        if not files:
            print(f"Keine Dateien im Ordner {folder_name} gefunden.")
            continue
            
        # Neuen Unterordner für die Ergebnisse erstellen
        output_folder = os.path.join(BASE_DIR, folder_name, "Extracted_Measurements", "Processed_with_Features")
        os.makedirs(output_folder, exist_ok=True)
        
        for f in files:
            filename = os.path.basename(f)
            print(f"Verarbeite: {filename}...")
            
            try:
                # Datenverarbeitung aufrufen
                processed_df = process_file(f, prefix)
                
                # Speichern im exakt gleichen Format wie die Originaldatei!
                # Da sep=',' und decimal=',' gesetzt sind, packt Pandas automatisch
                # alle Kommazahlen sauber in doppelte Anführungszeichen ("12,34").
                output_path = os.path.join(output_folder, filename)
                processed_df.to_csv(output_path, index=False, sep=',', decimal=',', quoting=csv.QUOTE_MINIMAL)
                
            except Exception as e:
                print(f"Fehler bei Datei {filename}: {e}")

if __name__ == "__main__":
    print("Starte Datenverarbeitung...")
    run_extraction()
    print("\nErfolgreich beendet! Die Originalstruktur wurde beibehalten.")

Starte Datenverarbeitung...
Verarbeite: output_10_A_c.csv...
Verarbeite: output_10_A_w.csv...
Verarbeite: output_11_A_c.csv...
Verarbeite: output_11_A_w.csv...
Verarbeite: output_12_A_c.csv...
Verarbeite: output_12_A_w.csv...
Verarbeite: output_13_A_c.csv...
Verarbeite: output_13_A_w.csv...
Verarbeite: output_14_A_c.csv...
Verarbeite: output_14_A_w.csv...
Verarbeite: output_15_A_c.csv...
Verarbeite: output_15_A_w.csv...
Verarbeite: output_16_A_c.csv...
Verarbeite: output_16_A_w.csv...
Verarbeite: output_17_A_c.csv...
Verarbeite: output_17_A_w.csv...
Verarbeite: output_18_A_c.csv...
Verarbeite: output_18_A_w.csv...
Verarbeite: output_19_A_c.csv...
Verarbeite: output_19_A_w.csv...
Verarbeite: output_1_A_c.csv...
Verarbeite: output_1_A_w.csv...
Verarbeite: output_2_A_c.csv...
Verarbeite: output_2_A_w.csv...
Verarbeite: output_3_A_c.csv...
Verarbeite: output_3_A_w.csv...
Verarbeite: output_4_A_c.csv...
Verarbeite: output_4_A_w.csv...
Verarbeite: output_5_A_c.csv...
Verarbeite: output_5_A_w